In [2]:
# mount google drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
# import necessary libraries
import pandas as pd
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import seaborn as sns
import glob

# As an inital example, we open one region and one model (aligned with chirps), to show the process of tercile analysis

In [29]:
# open merged nc file
ss_GFDL = xr.open_dataset('/content/drive/My Drive/capstone_data/netCDF/south_sudan_GFDL_merged.nc')

ss_GFDL_df = ss_GFDL.to_dataframe()

ss_GFDL_df = ss_GFDL_df.reset_index()

In [30]:
# take the ensemble mean
ss_GFDL_df = (ss_GFDL_df
                            .groupby(['time', 'lead_time', 'latitude', 'longitude'])[['predicted_precip', 'precip']]
                            .mean().reset_index())

# take the spatial means
ss_GFDL_df = (ss_GFDL_df  # Use the ensemble_means DataFrame
                              .groupby(['time', 'lead_time'])[['predicted_precip', 'precip']]
                              .mean().reset_index())

In [31]:
ss_GFDL_df['month'] = ss_GFDL_df['time'].dt.month
ss_GFDL_df['year'] = ss_GFDL_df['time'].dt.year
ss_GFDL_df = ss_GFDL_df.drop('time', axis=1, inplace=False)

ss_GFDL_df = ss_GFDL_df.reset_index().dropna()

In [32]:
# query a month and lead time to view tercile cutoff values
ss_GFDL_df.query('month == 2 and lead_time == 0.5').quantile([0.33, 0.66])

,index,lead_time,predicted_precip,precip,month,year
0.33,1532.64,0.5,0.195708,6.306736,2.0,2001.56
0.66,3053.28,0.5,0.435842,8.472741,2.0,2012.12


In [33]:
import pandas as pd

# Define some helper functions, then the main dataframe generation function

# Compute the Tercile Cutoffs for each unique month and lead_time
def get_tercile_cutoffs(df):
    return df.quantile([0.33, 0.66])

# Assign the Tercile Category for both predicted_precip and precip
def assign_tercile_category(value, lower_cutoff, upper_cutoff):
    if value <= lower_cutoff:
        return 'Low'
    elif value <= upper_cutoff:
        return 'Medium'
    else:
        return 'High'

# generate a tercile dataframe for one region and one model
def compute_tercile_monthly_df(merged_file_path_nc):

  # extract region and model name from netcdf path
  model = merged_file_path_nc.split('/')[-1].split('_')[-2]
  region = '_'.join(merged_file_path_nc.split('/')[-1].split('_')[0:-2])

  # print model, region
  print(f'Model: {model}')
  print(f'Region: {region}')

  # open netcdf file
  current_netcdf = xr.open_dataset(merged_file_path_nc)

  current_df = current_netcdf.to_dataframe()

  current_df = current_df.reset_index()

  # take the ensemble mean
  current_df = (current_df
                            .groupby(['time', 'lead_time', 'latitude', 'longitude'])[['predicted_precip', 'precip']]
                            .mean().reset_index())

  # take the spatial means
  current_df = (current_df  # Use the ensemble_means DataFrame
                              .groupby(['time', 'lead_time'])[['predicted_precip', 'precip']]
                              .mean().reset_index())

  # separate month and year into columns, drop time
  current_df['month'] = current_df['time'].dt.month
  current_df['year'] = current_df['time'].dt.year
  current_df = current_df.drop('time', axis=1, inplace=False)

  # reset index for wrangling
  current_df = current_df.reset_index().dropna()

  # subset to 1993 to 2024
  current_df = current_df.query('year >= 1993 and year <= 2024')

  # Initialize the `agreement` column
  current_df['agreement'] = None

  # Iterate over each unique combination of month and lead_time, over all years
  for (month, lead_time), group in current_df.groupby(['month', 'lead_time']):
      # Compute tercile cutoffs for predicted_precip and precip for this group (across all years)
      cutoffs = get_tercile_cutoffs(group[['predicted_precip', 'precip']])

      # Extract cutoffs for predicted_precip and precip separately
      lower_cutoff_predicted = cutoffs.loc[0.33, 'predicted_precip']
      upper_cutoff_predicted = cutoffs.loc[0.66, 'predicted_precip']

      lower_cutoff_precip = cutoffs.loc[0.33, 'precip']
      upper_cutoff_precip = cutoffs.loc[0.66, 'precip']

      # assign tercile categories for predicted_precip and precip
      group[f'{model}_tercile_class'] = group['predicted_precip'].apply(assign_tercile_category, args=(lower_cutoff_predicted, upper_cutoff_predicted))
      group['chirps_tercile_class'] = group['precip'].apply(assign_tercile_category, args=(lower_cutoff_precip, upper_cutoff_precip))

      # Calculate agreement: 1 if terciles agree, 0 otherwise
      group['agreement'] = (group[f'{model}_tercile_class'] == group['chirps_tercile_class']).astype(int)

      # Update the DataFrame with the new columns
      current_df.loc[group.index, f'{model}_tercile_class'] = group[f'{model}_tercile_class']
      current_df.loc[group.index, 'chirps_tercile_class'] = group['chirps_tercile_class']
      current_df.loc[group.index, 'agreement'] = group['agreement']
      current_df.loc[group.index, 'model'] = model
      current_df.loc[group.index, 'region'] = region

  # Select the desired columns for the final DataFrame
  final_columns = ['year', 'lead_time', 'month', 'predicted_precip', 'precip', f'{model}_tercile_class', 'chirps_tercile_class', 'agreement', 'model', 'region']
  final_df = current_df[final_columns]

  return final_df

In [34]:
# usage of main function
merged_file_path_nc = '/content/drive/My Drive/capstone_data/netCDF/south_sudan_GFDL_merged.nc'
ss_GFDL_tercile_df = compute_tercile_monthly_df(merged_file_path_nc)

# view dataset
ss_GFDL_tercile_df

Model: GFDL
Region: south_sudan


,year,lead_time,month,predicted_precip,precip,GFDL_tercile_class,chirps_tercile_class,agreement,model,region
0,1991,0.5,1,0.290611,5.262323,High,High,1,GFDL,south_sudan
12,1991,0.5,2,0.248649,9.689799,Medium,High,0,GFDL,south_sudan
13,1991,1.5,2,0.284590,9.689799,Medium,High,0,GFDL,south_sudan
24,1991,0.5,3,1.022405,26.528862,Low,Medium,0,GFDL,south_sudan
25,1991,1.5,3,1.851229,26.528862,High,Medium,0,GFDL,south_sudan
...,...,...,...,...,...,...,...,...,...,...
4891,2024,7.5,12,0.244016,4.464745,Low,Low,1,GFDL,south_sudan
4892,2024,8.5,12,0.392236,4.464745,Low,Low,1,GFDL,south_sudan
4893,2024,9.5,12,0.201560,4.464745,Low,Low,1,GFDL,south_sudan
4894,2024,10.5,12,0.235091,4.464745,Low,Low,1,GFDL,south_sudan


In [50]:
# query a month and lead time to view tercile cutoff values
ss_GFDL_df.query('month == 1 and lead_time == 0.5').quantile([0.33, 0.66])

,index,lead_time,predicted_precip,precip,month,year
0.33,1520.64,0.5,0.125227,3.297097,1.0,2001.56
0.66,3041.28,0.5,0.211919,4.878140,1.0,2012.12


### We see that for several months and lead times, the tercile cutoffs are different when averaged across all the years of interest. These cutoff values align with our tercile classifications. For example, January at a lead time of 0.5 has an upper cutoff of 0.211 for GFDL, and 4.8 for CHIRPS. In the dataframe, we see that both GFDL and CHIRPS classify the tercile as high, since the actual values were 0.29 and 5.26 respectively. As a result, the agreement is 1.

## Now, we group by month and lead time to view the agreement rate over all the years of interest (1993-2024).

In [35]:
# subset by chirps tercile class = low
ss_GFDL_df_low = ss_GFDL_tercile_df.query('chirps_tercile_class == "Low"')

# compute agreement rates
ss_GFDL_df_low.groupby(['month', 'lead_time', 'model', 'region'])[['agreement']].mean().reset_index()

,month,lead_time,model,region,agreement
0,1,0.5,GFDL,south_sudan,0.454545
1,1,1.5,GFDL,south_sudan,0.454545
2,1,2.5,GFDL,south_sudan,0.272727
3,1,3.5,GFDL,south_sudan,0.363636
4,1,4.5,GFDL,south_sudan,0.4
...,...,...,...,...,...
139,12,7.5,GFDL,south_sudan,0.363636
140,12,8.5,GFDL,south_sudan,0.363636
141,12,9.5,GFDL,south_sudan,0.363636
142,12,10.5,GFDL,south_sudan,0.272727


# After verifying the results for one region and one model, the function can be run on all the data

In [5]:
# takes 15 mins on colab

# Initialize an empty dictionary to store DataFrames
dfs_dict = {}

# initiate file list
list_of_files = glob.glob('/content/drive/My Drive/capstone_data/netCDF/*')

# Loop over all files
for f in list_of_files:
    # Generate the DataFrame
    df = compute_tercile_monthly_df(f)

    # Store the DataFrame in the dictionary with the year as key
    dfs_dict[f] = df

# Concatenate all DataFrames in the dictionary into one DataFrame
final_df = pd.concat(dfs_dict.values(), ignore_index=True)

# Now `final_df` contains all concatenated data

Model: CMCC
Region: west_africa
Model: CMCC
Region: southern_africa
Model: CMCC
Region: eastern_ukraine
Model: CMCC
Region: eastern_east_africa
Model: CMCC
Region: south_sudan
Model: NCEP
Region: lake_victoria_basin
Model: NCEP
Region: sri_lanka
Model: NCEP
Region: west_africa
Model: NCEP
Region: southern_africa
Model: NCEP
Region: eastern_ukraine
Model: METEO
Region: southern_africa
Model: METEO
Region: eastern_ukraine
Model: METEO
Region: eastern_east_africa
Model: METEO
Region: south_sudan
Model: ECMWF
Region: lake_victoria_basin
Model: ECMWF
Region: sri_lanka
Model: ECMWF
Region: west_africa
Model: ECMWF
Region: southern_africa
Model: ECMWF
Region: eastern_ukraine
Model: ECMWF
Region: eastern_east_africa
Model: ECMWF
Region: south_sudan
Model: DWD
Region: lake_victoria_basin
Model: DWD
Region: sri_lanka
Model: DWD
Region: west_africa
Model: DWD
Region: southern_africa
Model: DWD
Region: eastern_ukraine
Model: DWD
Region: eastern_east_africa
Model: DWD
Region: south_sudan
Model: CMC

## Subset the combined final dataframe into low, middle, and upper terciles for plotting

In [43]:
# subset the final df by tercile class, low, medium, and high for plotting

# subset by chirps tercile class = low
ss_GFDL_df_low = final_df.query('chirps_tercile_class == "Low"')

# compute agreement rates
ss_GFDL_df_low = ss_GFDL_df_low.groupby(['month', 'lead_time', 'model', 'region'])[['agreement']].mean().reset_index()

# subset by chirps tercile class = medium
ss_GFDL_df_medium = final_df.query('chirps_tercile_class == "Medium"')

# compute agreement rates
ss_GFDL_df_medium = ss_GFDL_df_medium.groupby(['month', 'lead_time', 'model', 'region'])[['agreement']].mean().reset_index()

# subset by chirps tercile class = High
ss_GFDL_df_high = final_df.query('chirps_tercile_class == "High"')

# compute agreement rates
ss_GFDL_df_high = ss_GFDL_df_high.groupby(['month', 'lead_time', 'model', 'region'])[['agreement']].mean().reset_index()

In [48]:
# plot the dataframe heatmap in seaborn
# done one at a time so far, manually changing variable names etc.
# set empty values to skyblue
sns.set(rc={'axes.facecolor':'skyblue'})

# set png resolution
plt.rcParams['figure.dpi'] = 300

# Clean the final dataframe
df_clean = ss_GFDL_df_medium.dropna(subset=['model', 'region']).copy()

# Convert integer lead times to 0.5 increments (e.g., 1 -> 0.5)
# this is how it solves for lead times 1 to 6
df_clean['lead_time'] = df_clean['lead_time'].apply(
    lambda x: x - 0.5 if x % 1 == 0 else x
)

## get the names for the models
models = df_clean['model'].unique()

# get the regions
regions = df_clean['region'].unique()

# define months and lead times range
months = range(1, 13)
lead_times = np.arange(0.5, 12, 1)  # 0.5 to 11.5 in steps of 1

# Generate grid parameters
lead_times = np.arange(0.5, 12, 1)  # Fixed grid from 0.5-11.5 in 1.0 steps

# Create complete grid
all_combinations = pd.MultiIndex.from_product(
    [models, regions, months, lead_times],
    names=['model', 'region', 'month', 'lead_time']
).to_frame(index=False)

# Merge with data
merged_df = all_combinations.merge(
    df_clean[['model', 'region', 'month', 'lead_time', 'agreement']],
    on=['model', 'region', 'month', 'lead_time'],
    how='left'
)

# Create FacetGrid and plot as before
g = sns.FacetGrid(
    merged_df,
    row='model',
    col='region',
    sharex=False,
    sharey=False
)

def draw_heatmap(data, **kwargs):
    # Aggregate data to remove duplicates
    data = data.groupby(['month', 'lead_time'], as_index=False)['agreement'].mean()

    # Pivot data
    pivot_data = data.pivot(index='month', columns='lead_time', values='agreement')
    pivot_data = pivot_data.reindex(index=months, columns=lead_times)

    # Convert to float and fill NaNs with a placeholder (e.g., 0 or np.nan)
    pivot_data = pivot_data.astype(float).fillna(np.nan)

    sns.heatmap(
        pivot_data,
        vmin=0,
        vmax=1,
        cmap=sns.color_palette("RdYlGn", 10),
        annot=False,
        cbar=True,
        square=True,
        linewidths=0.5,
        linecolor='black'
    )
    plt.gca().invert_yaxis()
    plt.xticks(ticks=np.arange(len(lead_times)) + 0.5, labels=lead_times, rotation=0)
    plt.xticks(fontsize=6)
    plt.yticks(fontsize=6)
    plt.yticks(ticks=np.arange(len(months)) + 0.5, labels=months, rotation=0)


g.map_dataframe(draw_heatmap)
g.set_axis_labels('Lead Time', 'Month')
g.set_titles('Tercile Hit Rate \n Region={col_name} \n Model={row_name} \n Tercile=medium')

# save and show figures
# plt.savefig('/content/drive/My Drive/capstone_data/figures/medium_tercile_hit_rate_monthly.png')
# plt.show()

## Interpretation of a graph

For January at lead time 0.5, tercile cutoff values are computed for observed and predicted precipitation. Then, the terciles are classified as 'Low', 'Medium", and 'High'. This is done for every month and lead time combination. Then, we find which years are in each tercile for both predicted and observed, then we find what the agreement rate between CHIRPS and any given model is.

For example, if the value for Lower Tercile GFDL South Sudan January at lead time 0.5 is 0.6, that means between 1993-2024, for specifically January across 1993-2024, GFDL's 0.5 lead prediction agrees with CHIRPS' classification of low precipitation 60% of the time.